In [63]:
import pandas as pd

In [64]:
# Read in the data
df = pd.read_csv("../data/data.csv")

In [65]:
#clean job titles by removing special characters and converting to lowercase. keep spaces
df['job_title'] = df['job_title'].str.replace('[^a-zA-Z0-9\\s]', '', regex=True).str.lower()

#remove extra whitespace, newlines, and carriage returns from job titles
df['job_title'] = df['job_title'].str.replace(r'[\r\n\s]+', ' ', regex=True)

print(df["job_title"].value_counts())


job_title
2019 ct bauer college of business graduate magna cum laude and aspiring human resources professional                   7
aspiring human resources professional                                                                                  7
student at humber college and aspiring human resources generalist                                                      7
people development coordinator at ryan                                                                                 6
native english teacher at epik english program in korea                                                                5
aspiring human resources specialist                                                                                    5
hr senior specialist                                                                                                   5
advisory board member at celal bayar university                                                                        4
seeking human resource

In [66]:
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Shane\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [67]:
# join all job titles into a single string and split into words
all_titles = ' '.join(df['job_title'].dropna())
words = all_titles.split()

#remove stop words from the word counts
words_no_stopwords = [word for word in words if word not in stop_words]

def get_common_words(words, n=20):
    from collections import Counter
    word_counts = Counter(words)
    common_words = pd.DataFrame(word_counts.most_common(n), columns=['word', 'count'])
    return common_words

In [68]:
common_words_no_stopwords = get_common_words(words_no_stopwords, n=20)
common_words_no_stopwords


,word,count
0,human,63
1,resources,63
2,aspiring,35
3,professional,20
4,student,16
5,seeking,15
6,college,14
7,generalist,14
8,university,12
9,specialist,12


In [69]:
#find common phrases in job titles to the nth-gram
from sklearn.feature_extraction.text import CountVectorizer

def find_ngrams(texts, n):
    vectorizer = CountVectorizer(ngram_range=(n, n))
    
    X = vectorizer.fit_transform(texts)
    
    ngram_df = pd.DataFrame({
        'phrase': vectorizer.get_feature_names_out(),
        'count': X.sum(axis=0).A1 #A1: flatten the array to 1D
    })
    
    return ngram_df.sort_values('count', ascending=False)

In [70]:
common_phrases_2 = find_ngrams(df['job_title'].dropna(), 3)

common_phrases_2.head(50)

,phrase,count
22,aspiring human resources,35
112,human resources professional,19
14,and aspiring human,16
104,human resources generalist,10
214,seeking human resources,9
0,2019 ct bauer,7
45,business graduate magna,7
32,at humber college,7
41,bauer college of,7
53,college and aspiring,7


In [71]:
#Example list of jobs
jobs = [
    "Software Engineer",
    "Data Scientist",
    "Product Manager",
    "Human Resources",
]

In [ ]:
#preprocessing location column

#two formats of location: "Region" and "City, Country"

has_city_country = df["location"].str.contains(",", na=False)
split_location = df.loc[has_city_country, "location"].str.split(",", n=1, expand=True)

df["city"] = pd.NA
df["region"] = pd.NA
df["country"] = pd.NA

df.loc[has_city_country, "city"] = split_location[0].str.strip()
df.loc[has_city_country, "region"] = split_location[1].str.strip()


is_region = ~df["location"].str.contains(",", na=False) & (df["location"] != "Kanada")
df.loc[is_region, "region"] = df.loc[is_region, "location"]

is_canada = df["location"].str.contains("Kanada", na=False)
is_usa = ~is_canada
df.loc[is_canada, "country"] = "Canada"
df.loc[is_usa, "country"] = "USA"

# handle all null values the same way
df["city"] = df["city"].astype("string")
df["country"] = df["country"].astype("string")
df["region"] = df["region"].astype("string")

df

,id,job_title,location,connection,fit,city,region,country
0,1,2019 ct bauer college of business graduate mag...,"Houston, Texas",85,NaN,Houston,Texas,USA
1,2,native english teacher at epik english program...,Kanada,500+,NaN,<NA>,<NA>,Canada
2,3,aspiring human resources professional,"Raleigh-Durham, North Carolina Area",44,NaN,Raleigh-Durham,North Carolina Area,USA
3,4,people development coordinator at ryan,"Denton, Texas",500+,NaN,Denton,Texas,USA
4,5,advisory board member at celal bayar university,"İzmir, Türkiye",500+,NaN,İzmir,Türkiye,USA
...,...,...,...,...,...,...,...,...
99,100,aspiring human resources manager graduating ma...,"Cape Girardeau, Missouri",103,NaN,Cape Girardeau,Missouri,USA
100,101,human resources generalist at loparex,"Raleigh-Durham, North Carolina Area",500+,NaN,Raleigh-Durham,North Carolina Area,USA
101,102,business intelligence and analytics at travelers,Greater New York City Area,49,NaN,<NA>,Greater New York City Area,USA
102,103,always set them up for success,Greater Los Angeles Area,500+,NaN,<NA>,Greater Los Angeles Area,USA


In [ ]:
"""to do:

identify professions
identify education classes
identify seniority

"""